# 1.install and import packges 

In [1]:
!pip install -qU \
    langchain \
    langchain-community \
    langchain-groq \
    pypdf \
    fastembed \
    chromadb \
    -qU langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 6.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.5 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 68.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.8/572.8 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 51.1 MB/s eta 0:00:00:00:0100:

In [2]:
import langchain
import langchain_groq
import pypdf
import fastembed
import chromadb
from langchain_community.document_loaders import PyPDFLoader
import os

print("All packages installed successfully!")

All packages installed successfully!


/tmp/ipykernel_49/481779447.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


=======================================================\
pypdf : to load a pdf \
fastembde : to use the embedding \
chromdb : to store the embedding 


# 2.import the llm and the embedding model 

In [17]:
from kaggle_secrets import UserSecretsClient
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_groq import ChatGroq
import os
#llm
user_secrets = UserSecretsClient()
GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY


class Models:
    def __init__(
        self,
        Chat_model="openai/gpt-oss-20b",
        embedding_model="thenlper/gte-large"
    ):
        self.Chat_model = Chat_model
        self.embedding_model_name = embedding_model

        self.chat_llm = ChatGroq(
            model=self.Chat_model,
            temperature=0.8,
            max_tokens=2000,
        )

        self.embedding_model = FastEmbedEmbeddings(
            model_name=self.embedding_model_name
        )

# load the pdf 

In [18]:
class File_Loader:
    ALLOWED_FILES = ["pdf"]

    def __init__(self, File_links):
        self.File_links = File_links

    def get_file_type(self, file_link):
        extension = os.path.splitext(file_link)[1].lower().replace(".", "")

        if extension in self.ALLOWED_FILES:
            return extension

        return "none"

    def load_files(self):
        files = []

        for file_link in self.File_links:
            file_type = self.get_file_type(file_link)

            files.append({
                "path": file_link,
                "type": file_type
            })

        return files

    def load_allowed_files(self):
        documents = []

        files = self.load_files()

        for file in files:
            if file["type"] == "pdf":
                loader = PyPDFLoader(file["path"])
                documents.extend(loader.load())

        return documents

file_links = [
    "/kaggle/input/datasets/mohamedjahha/pdf-test/lekl101.pdf",
    "/kaggle/input/datasets/mohamedjahha/pdf-test/lekl101.txt",
    "/kaggle/input/datasets/mohamedjahha/pdf-test/test.docx"
]

loader = File_Loader(file_links)

documents = loader.load_allowed_files()

print(f"Loaded documents: {len(documents)}")

Loaded documents: 11


# splitting the file 

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

class Chunker:
    def __init__(self, File_links, chunk_size=1000, chunk_overlap=150):

        # Chunker internally uses File_Loader
        self.file_loader = File_Loader(File_links)

        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

    def split(self):
        # Load allowed files
        documents = self.file_loader.load_allowed_files()

        # Split documents into chunks
        return self.splitter.split_documents(documents)

file_links = [
    "/kaggle/input/datasets/mohamedjahha/pdf-test/lekl101.pdf",
    "/kaggle/input/datasets/mohamedjahha/pdf-test/lekl101.txt",
    "/kaggle/input/datasets/mohamedjahha/pdf-test/test.docx"
]

chunker = Chunker(file_links)

chunks = chunker.split()

print(f"Number of chunks: {len(chunks)}")

Number of chunks: 27


# embedding 

In [20]:
from langchain_chroma import Chroma

class Embedding:
    def __init__(
        self,
        File_links,
        embedding_model="thenlper/gte-large",
        chat_model="openai/gpt-oss-20b",
        persist_directory="./chroma_db",
        chunk_size=1000,
        chunk_overlap=150
    ):
        self.File_links = File_links
        self.persist_directory = persist_directory

        # Use LLM class
        self.Models = Models(
            Chat_model=chat_model,
            embedding_model=embedding_model
        )

        # Get embedding model from LLM
        self.embedding_model = self.Models.embedding_model

        # Get chat model from LLM
        self.chat_llm = self.Models.chat_llm

        # Chunker internally uses File_Loader
        self.chunker = Chunker(
            File_links,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

        # Create chunks
        self.chunks = self.chunker.split()

        # Create ChromaDB
        self.db = Chroma.from_documents(
            documents=self.chunks,
            embedding=self.embedding_model,
            persist_directory=self.persist_directory
        )

    def get_db(self):
        return self.db

    def get_chunks(self):
        return self.chunks

    def get_Models(self):
        return self.chat_Models


In [21]:
query = "What is the importance of the title 'I Sell My Dreams ?"

# retriever part [we will use the mmr]

In [36]:
class Retriever:
    def __init__(
        self,
        embedding,
        search_type="mmr",
        k=4
    ):
        self.db = embedding.get_db()

        self.retriever = self.db.as_retriever(
            search_type=search_type,
            search_kwargs={"k": k}
        )

    def retrieve(self, query):
        return self.retriever.invoke(query)

    def get_retriever(self):
        return self.retriever

# RAG 

In [43]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


class RAG_Retrieval:

    def __init__(self, file_links):
        # VectorDB is created ONCE
        self.embedding = Embedding(
            File_links=file_links
        )

    def get_retriever(self, k=4):
        retriever = self.embedding.get_db().as_retriever(
            search_type="mmr",
            search_kwargs={"k": k}
        )

        return retriever



class RAG_Generation:

    def __init__(
        self,
        retrieval,
        k=4,
        template="""
You are an AI assistant that answers questions using the provided context.

Rules:

- If the context does not contain enough information to answer the question,
  say exactly: "I don't know."
- if you have any information that can help say it 

Context:
{context}

Question:
{query}

Answer:
"""
    ):

        self.retrieval = retrieval
        self.k = k

        self.models = Models()

        self.prompt = ChatPromptTemplate.from_template(template)

        self.retriever = self.retrieval.get_retriever(
            k=self.k
        )

        self.chain = (
            {
                "context": self.retriever,
                "query": RunnablePassthrough()
            }
            | self.prompt
            | self.models.chat_llm
            | StrOutputParser()
        )

    def response(self, query):
        return self.chain.invoke(query)

In [38]:
file_links = [
    "/kaggle/input/datasets/mohamedjahha/pdf-test/lekl101.pdf",
    "/kaggle/input/datasets/mohamedjahha/pdf-test/lekl101.txt",
    "/kaggle/input/datasets/mohamedjahha/pdf-test/test.docx"
]

In [39]:
rag_retrieval = RAG_Retrieval(file_links)

In [44]:
rag = RAG_Generation(
    rag_retrieval,
    k=8
)

print(rag.response(
    "What is the significance of the title 'I Sell My Dreams'?"
))

The title “I Sell My Dreams” frames the story as a kind of marketplace for the narrator’s imagination.  In the narrative the narrator is a poet who meets a woman, Frau Frieda, who literally “sold” her property and whose life is built on dreams.  The poet’s own dreams—the stories, images and insights he wishes to share—are offered to others in the same way, as if they were a commodity.  Thus the title signals the central tension of the piece: the commodification of the imaginative, the idea that dreams can be bought, sold and traded, and the way the narrator’s creative life is shaped by this exchange.  In short, the title underlines that the narrator is selling his dreams—his creative output—to the world, and that this act of selling is what drives the story’s exploration of art, commerce and the power of storytelling.
